In [ ]:
# ==============================================================================
# GALACTIC ROTATION CURVE TEST: Binary (NFW) vs Ternary (NFW + Gabor S₃)
# ==============================================================================
#
# SAME FRAMEWORK AS THE POLYMARKET TEST:
# - Real data (175 galaxies from SPARC)
# - The ternary model CAN LOSE (BIC penalizes extra parameters)
# - The universe decides
#
# BINARY MODEL: Baryonic matter + NFW dark matter halo
# TERNARY MODEL: Baryonic matter + NFW halo + S₃ Gabor uncertainty envelope
#
# The S₃ component represents structure in the conjugate domain that
# binary measurement systems (position OR momentum) trade away.
# Its width is constrained by a Gabor-like relation to the baryonic
# scale length: σ_S3 must satisfy σ_r × σ_k ≥ 1/(4π)
#
# If S₃ adds noise, BIC kills it. If it captures real structure, it wins.
# ==============================================================================

import numpy as np
from scipy.optimize import least_squares
import zipfile
import io
import os
import urllib.request

# ------------------------------------------------------------------------------
# STEP 1: Download SPARC data
# ------------------------------------------------------------------------------

SPARC_URL = "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip"
DATA_DIR = "sparc_data"

def download_sparc():
    if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 100:
        print(f"SPARC data already exists ({len(os.listdir(DATA_DIR))} files)")
        return

    print("Downloading SPARC rotation curve data...")
    response = urllib.request.urlopen(SPARC_URL)
    zip_data = io.BytesIO(response.read())

    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_data) as zf:
        zf.extractall(DATA_DIR)

    print(f"Downloaded and extracted to {DATA_DIR}/")

download_sparc()

# ------------------------------------------------------------------------------
# STEP 2: Parse SPARC galaxy files
# ------------------------------------------------------------------------------

def parse_galaxy(filepath):
    """
    Parse a SPARC rotation curve file.
    Columns: Rad(kpc) Vobs(km/s) errV Vgas Vdisk Vbul SBdisk SBbul
    """
    data = {"Rad": [], "Vobs": [], "errV": [], "Vgas": [], "Vdisk": [], "Vbul": []}

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('!'):
                continue
            parts = line.split()
            if len(parts) < 6:
                continue
            try:
                vals = [float(p) for p in parts[:6]]
                data["Rad"].append(vals[0])
                data["Vobs"].append(vals[1])
                data["errV"].append(max(vals[2], 1.0))  # floor errors at 1 km/s
                data["Vgas"].append(vals[3])
                data["Vdisk"].append(vals[4])
                data["Vbul"].append(vals[5])
            except ValueError:
                continue

    return {k: np.array(v) for k, v in data.items()}

def load_all_galaxies():
    """Load all parseable SPARC galaxies with enough data points."""
    galaxies = {}

    # Find the actual data files (may be in subdirectory)
    search_dirs = [DATA_DIR]
    for root, dirs, files in os.walk(DATA_DIR):
        search_dirs.append(root)

    for search_dir in search_dirs:
        for fname in sorted(os.listdir(search_dir)):
            fpath = os.path.join(search_dir, fname)
            if not os.path.isfile(fpath):
                continue
            # SPARC files are named like "NGC2403_rotmod.dat" or similar
            if not (fname.endswith('.dat') or fname.endswith('.txt') or '.' not in fname
                    or fname.startswith('NGC') or fname.startswith('UGC')
                    or fname.startswith('DDO') or fname.startswith('IC')
                    or fname.startswith('ESO') or fname.startswith('F')):
                # Try parsing anyway if it looks like a data file
                pass

            try:
                gal = parse_galaxy(fpath)
                if len(gal["Rad"]) >= 8:  # need enough points for fitting
                    name = fname.replace('_rotmod.dat', '').replace('.dat', '')
                    galaxies[name] = gal
            except:
                continue

    print(f"Loaded {len(galaxies)} galaxies with >= 8 data points")
    return galaxies

galaxies = load_all_galaxies()

if len(galaxies) == 0:
    # Try listing what's actually in the directory
    print("\nDEBUG: Contents of data directory:")
    for root, dirs, files in os.walk(DATA_DIR):
        for f in files[:20]:
            print(f"  {os.path.join(root, f)}")
    print("  ...")
    raise SystemExit("Could not parse galaxy files — check directory structure above")

# Show a few examples
for name in list(galaxies.keys())[:3]:
    g = galaxies[name]
    print(f"  {name}: {len(g['Rad'])} points, R=[{g['Rad'][0]:.1f}-{g['Rad'][-1]:.1f}] kpc, "
          f"Vobs=[{g['Vobs'].min():.0f}-{g['Vobs'].max():.0f}] km/s")

# ------------------------------------------------------------------------------
# STEP 3: Define velocity models
# ------------------------------------------------------------------------------

def V_baryonic(gal, Ydisk=0.5, Ybul=0.7):
    """
    Total baryonic velocity contribution.
    V_bar² = Ydisk * Vdisk² + Vgas² + Ybul * Vbul²

    Note: SPARC Vdisk and Vbul can be negative (indicating the sign of the
    gravitational contribution in the original mass model). We use the
    convention V² = sign(V) * V² to preserve this.
    """
    Vd2 = Ydisk * np.sign(gal["Vdisk"]) * gal["Vdisk"]**2
    Vg2 = np.sign(gal["Vgas"]) * gal["Vgas"]**2
    Vb2 = Ybul * np.sign(gal["Vbul"]) * gal["Vbul"]**2

    Vbar2 = Vd2 + Vg2 + Vb2
    return Vbar2  # return V² (can be negative at small radii)


def V_NFW_squared(r, rho0, rs):
    """
    NFW halo velocity squared: V²(r) for Navarro-Frenk-White profile.

    V²_NFW(r) = (4π G ρ₀ rs³ / r) × [ln(1 + r/rs) - (r/rs)/(1 + r/rs)]

    G = 4.302e-3 pc (km/s)² / M_sun
    We absorb constants into V200² parameter for numerical stability.
    """
    x = r / rs
    # Avoid division by zero
    x = np.maximum(x, 1e-10)

    # NFW enclosed mass function: f(x) = ln(1+x) - x/(1+x)
    fx = np.log(1 + x) - x / (1 + x)

    # V²_NFW = V200² × (1/x) × f(x) / f(c)
    # We parameterize with Vmax² and rs directly for fitting simplicity
    # V² = Vscale² × f(x) / x  where Vscale absorbs all constants
    Vscale_sq = rho0  # rho0 here is really Vscale² (km/s)² for numerical ease

    return Vscale_sq * fx / x


def V_S3_squared(r, V_s3, r_peak, sigma_g):
    """
    S₃ Gabor uncertainty envelope: structure in the conjugate domain.

    A Gaussian velocity contribution centered at r_peak with width σ_g.
    The Gabor constraint: σ_g is related to the baryonic scale length.

    V²_S3(r) = V_s3² × exp(-(r - r_peak)² / (2σ_g²))
    """
    return V_s3 * np.exp(-0.5 * ((r - r_peak) / sigma_g)**2)


# ------------------------------------------------------------------------------
# STEP 4: Fitting functions
# ------------------------------------------------------------------------------

def fit_binary(gal, Ydisk=0.5, Ybul=0.7):
    """
    Fit binary model: V²_total = V²_bar + V²_NFW
    Free params: Vscale² (NFW amplitude), rs (NFW scale radius)
    """
    r = gal["Rad"]
    Vobs = gal["Vobs"]
    errV = gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)

    def residuals(params):
        log_Vscale2, log_rs = params
        Vscale2 = 10**log_Vscale2
        rs = 10**log_rs

        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vtot2 = Vbar2 + Vnfw2

        # V_total = sqrt(V²) but handle negative V²
        Vtot = np.sign(Vtot2) * np.sqrt(np.abs(Vtot2))

        return (Vtot - Vobs) / errV

    # Initial guess
    x0 = [4.0, 0.5]  # log10(Vscale²) ~ 10000, log10(rs) ~ 3 kpc
    bounds = ([1, -1], [7, 2.5])  # broad bounds

    try:
        result = least_squares(residuals, x0, bounds=bounds, method='trf', max_nfev=5000)
        Vscale2 = 10**result.x[0]
        rs = 10**result.x[1]

        chi2 = np.sum(result.fun**2)
        n = len(r)
        k = 2  # number of free params
        bic = chi2 + k * np.log(n)

        return {"chi2": chi2, "bic": bic, "n": n, "k": k,
                "params": {"Vscale2": Vscale2, "rs": rs}, "success": result.success}
    except:
        return None


def fit_ternary(gal, Ydisk=0.5, Ybul=0.7):
    """
    Fit CONSTRAINED ternary model: V²_total = V²_bar + V²_NFW + V²_S3

    S₃ is COUPLED to baryonic structure (not floating):
      - r_peak = 2.15 × h_R  (where disk velocity peaks)
      - σ_g = h_R             (width = disk scale length)

    This reduces free params to 3: Vscale² (NFW), rs (NFW), V_s3 (S3 amplitude)
    Only ONE more parameter than binary. BIC is nearly a fair fight.
    """
    r = gal["Rad"]
    Vobs = gal["Vobs"]
    errV = gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)

    # --- PHYSICAL COUPLING: Extract h_R from Vdisk profile ---
    # For an exponential disk, Vdisk peaks at R_peak ≈ 2.15 × h_R
    idx_max_disk = np.argmax(np.abs(gal["Vdisk"]))
    r_peak_disk = r[idx_max_disk]
    h_R = max(r_peak_disk / 2.15, 0.5)  # floor at 0.5 kpc for stability

    # Constrain S₃ to baryonic structure
    r_s3 = 2.15 * h_R    # centered on disk peak
    sigma_s3 = h_R        # width = disk scale length

    def residuals(params):
        # Only 3 free parameters — same as binary + 1
        log_Vscale2, log_rs, log_Vs3 = params
        Vscale2 = 10**log_Vscale2
        rs = 10**log_rs
        Vs3 = 10**log_Vs3

        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        # S₃ is anchored — no floating location or width
        Vs3_2 = V_S3_squared(r, Vs3, r_s3, sigma_s3)
        Vtot2 = Vbar2 + Vnfw2 + Vs3_2

        Vtot = np.sign(Vtot2) * np.sqrt(np.abs(Vtot2))

        return (Vtot - Vobs) / errV

    x0 = [4.0, 0.5, 3.0]
    bounds = ([1, -1, 0], [7, 2.5, 7])

    try:
        result = least_squares(residuals, x0, bounds=bounds, method='trf', max_nfev=5000)

        chi2 = np.sum(result.fun**2)
        n = len(r)
        k = 3  # only 1 more than binary
        bic = chi2 + k * np.log(n)

        return {"chi2": chi2, "bic": bic, "n": n, "k": k,
                "params": {"Vscale2": 10**result.x[0], "rs": 10**result.x[1],
                           "Vs3": 10**result.x[2], "h_R": h_R,
                           "r_s3": r_s3, "sigma_s3": sigma_s3},
                "success": result.success}
    except:
        return None


# ------------------------------------------------------------------------------
# STEP 5: Run the test across all galaxies
# ------------------------------------------------------------------------------

print(f"\n{'='*70}")
print(f"  GALACTIC ROTATION CURVE TEST")
print(f"  Binary (Baryonic + NFW) vs Ternary (Baryonic + NFW + Gabor S₃)")
print(f"  {len(galaxies)} SPARC galaxies — BIC model comparison")
print(f"{'='*70}\n")

results = []
binary_wins = 0
ternary_wins = 0
failures = 0

for name, gal in sorted(galaxies.items()):
    b = fit_binary(gal)
    t = fit_ternary(gal)

    if b is None or t is None or not b["success"] or not t["success"]:
        failures += 1
        continue

    delta_bic = t["bic"] - b["bic"]  # negative = ternary wins
    winner = "TERNARY" if delta_bic < -2 else ("BINARY" if delta_bic > 2 else "TIE")

    if winner == "TERNARY":
        ternary_wins += 1
    elif winner == "BINARY":
        binary_wins += 1

    results.append({
        "name": name,
        "n_points": b["n"],
        "binary_chi2": b["chi2"],
        "ternary_chi2": t["chi2"],
        "binary_bic": b["bic"],
        "ternary_bic": t["bic"],
        "delta_bic": delta_bic,
        "winner": winner,
    })

# Sort by delta BIC (most ternary-favorable first)
results.sort(key=lambda x: x["delta_bic"])

# ------------------------------------------------------------------------------
# STEP 6: Results
# ------------------------------------------------------------------------------

print(f"{'Galaxy':<20} {'N':>4} {'χ²_bin':>10} {'χ²_ter':>10} {'ΔBIC':>10} {'Winner':>10}")
print("-" * 70)

for r in results[:15]:  # Top 15 ternary-favorable
    print(f"{r['name']:<20} {r['n_points']:>4} {r['binary_chi2']:>10.1f} "
          f"{r['ternary_chi2']:>10.1f} {r['delta_bic']:>+10.1f} {r['winner']:>10}")

if len(results) > 30:
    print(f"  ... ({len(results) - 30} galaxies omitted) ...")

for r in results[-15:]:  # Top 15 binary-favorable
    print(f"{r['name']:<20} {r['n_points']:>4} {r['binary_chi2']:>10.1f} "
          f"{r['ternary_chi2']:>10.1f} {r['delta_bic']:>+10.1f} {r['winner']:>10}")

# Summary
ties = len(results) - binary_wins - ternary_wins
total = len(results)

print(f"\n{'='*70}")
print(f"  FINAL VERDICT — {total} galaxies fitted ({failures} fit failures)")
print(f"{'='*70}")
print(f"  Ternary wins (ΔBIC < -2): {ternary_wins} ({100*ternary_wins/total:.1f}%)")
print(f"  Binary wins  (ΔBIC > +2): {binary_wins} ({100*binary_wins/total:.1f}%)")
print(f"  Ties         (|ΔBIC| ≤ 2): {ties} ({100*ties/total:.1f}%)")

median_delta = np.median([r["delta_bic"] for r in results])
mean_delta = np.mean([r["delta_bic"] for r in results])

print(f"\n  Median ΔBIC: {median_delta:+.2f}")
print(f"  Mean ΔBIC:   {mean_delta:+.2f}")

if ternary_wins > binary_wins and median_delta < -2:
    print(f"\n  >>> S₃ GABOR ENVELOPE CAPTURES REAL STRUCTURE")
    print(f"  >>> The ternary model fits galactic rotation curves better")
    print(f"  >>> even after BIC penalizes the extra parameters.")
    print(f"  >>> The conjugate-domain structure is REAL.")
elif ternary_wins > binary_wins:
    print(f"\n  >>> Ternary wins more galaxies but effect is modest")
    print(f"  >>> Further investigation needed (e.g., galaxy subsamples)")
elif binary_wins > ternary_wins:
    print(f"\n  >>> BINARY (NFW) WINS — the S₃ component is overfitting")
    print(f"  >>> The Gabor envelope adds parameters without adding signal")
    print(f"  >>> HYPOTHESIS REJECTED for galactic rotation curves")
else:
    print(f"\n  >>> INCONCLUSIVE — roughly even split")
    print(f"  >>> May need more constrained S₃ parameterization")

print(f"\n  NOTE: S₃ is PHYSICALLY CONSTRAINED — coupled to baryonic h_R.")
print(f"  Binary: 2 free params. Ternary: 3 free params (only +1).")
print(f"  BIC penalty difference is minimal — ternary must EARN its place")
print(f"  through genuine χ² improvement, not parameter flexibility.")
print(f"  A proper follow-up would use MCMC with Bayes factors.")
print(f"{'='*70}")

Downloaded and extracted to sparc_data/
Loaded 143 galaxies with >= 8 data points
  CamB: 9 points, R=[0.2-1.8] kpc, Vobs=[2-20] km/s
  D631-7: 16 points, R=[0.5-7.2] kpc, Vobs=[8-58] km/s
  DDO064: 14 points, R=[0.1-3.0] kpc, Vobs=[6-47] km/s

  GALACTIC ROTATION CURVE TEST
  Binary (Baryonic + NFW) vs Ternary (Baryonic + NFW + Gabor S₃)
  143 SPARC galaxies — BIC model comparison

Galaxy                  N     χ²_bin     χ²_ter       ΔBIC     Winner
----------------------------------------------------------------------
UGC06787               71     1772.5     1101.1     -667.1    TERNARY
UGC02953              115      641.9      496.9     -140.3    TERNARY
NGC6674                15      176.9       77.0      -97.2    TERNARY
UGC02916               43     1839.7     1771.8      -64.1    TERNARY
NGC6015                44      347.9      280.5      -63.6    TERNARY
ESO563-G021            30      577.7      517.9      -56.4    TERNARY
NGC2841                50      160.3      105.2      